### 数据清洗，去重
#### 构建知识库重要的点
#### 1、数据多样性,数据要更符合实际情况，不要过度清洗，比如地点存在北京、北京市、beijing三种格式，就要保留三种格式。
#### 2、数据纯净性
#### 3、数据范围广
#### 4、数据...

In [6]:

# JD数据清洗
# 原则：1、正则去除杂质文件 2、
import pandas as pd
from pathlib import Path
import re
from bs4 import BeautifulSoup

# 汇总所有数据
def get_all_jd(path):
    folder = Path(path)
    files = [f for f in folder.iterdir() if f.is_file() and f.suffix.lower() in {'.xls', '.xlsx'}]
    df_list = []
    for f in files:
        # print(f)
        df_temp = pd.read_excel(f)
        df_temp['公司'] = str(f).split('\\')[1][:-5]
        df_list.append(df_temp[['分类','职位名称','工作地点','职位描述','职位要求','发布时间','职位类别名称','公司']])
    jd_df = pd.concat(df_list, axis=0, ignore_index=True)
    return jd_df

def dealStr(cell):
    parts = []
    if pd.notna(cell):
        if isinstance(cell, str) and cell.startswith('['):
            cell = eval(cell)
        if isinstance(cell, str):
            parts = re.split(r'[，、；;]+', cell)
        else:
            parts = cell
        if not isinstance(parts,list):
            print(parts)
            print(cell)
        return ','.join(parts)
    else:
        return ''

def cleanStr(cell: str) -> str:
    text = BeautifulSoup(cell, 'lxml').get_text(separator='')
    has_dot   = re.search(r'[•●⁃⁎∙]', text)          # 小圆点
    has_other = re.search(r'(?:\d{1,2}|[a-zA-Z])[、.．\-–—)\]]', text)  # 1. 2、 3) 等
    already_ok = re.search(r'^1、', text.strip()) and re.search(r'\n\d+、', text)

    # 如果不需要处理，直接返回原字符串
    if not (has_dot or has_other) or already_ok:
        return cell

    # 3. 真正需要处理：把「小圆点」先统一换成换行
    text = re.sub(r'[•●⁃⁎∙]\s*', '\n', text)

    # 4. 按「数字/字母 + 序号符号」切分
    pattern = re.compile(r'(?:\d{1,2}|[a-zA-Z])[、.．\-–—)\]]')
    parts = pattern.split(text)
    marks = pattern.findall(text)

    # 5. 重新编号成 1、2、3、
    out = []
    for idx, (m, seg) in enumerate(zip(marks, parts[1:]), 1):
        seg = seg.strip()
        if seg:
            out.append(f"{idx}、{seg}")

    # 6. 返回处理后的纯文本（不再带 HTML）
    return '\n'.join(out)

def target_job(job_name,job_requ,job_desc,job_date):
    hit = 0
    if re.search(r'数据|大模型',job_name):
        hit = hit + 1
    if job_date >= pd.Timestamp('2025-09-01'):
        hit = hit + 1
    if re.search(r'语料|大预言模型|数据集|LLM|全链路|数据清洗|数据评估|质量', job_requ):
        hit = hit + 1
    if re.search(r'语料|大预言模型|数据集|LLM|全链路|数据清洗|数据评估|质量', job_desc):
        hit = hit + 1
    if hit == 4:
        return 'hit'
    else:
        return 'not hit'
# 统一码值
def preprocess(df):
    df['发布时间'] = pd.to_datetime(df['发布时间'], errors='coerce')
    df = df[df['发布时间'] >= '2025-09-01']
    df = df[df['职位名称'].str.contains('数据|大模型', na=False)]
    df['工作地点'] = df['工作地点'].apply(dealStr)
    df['职位类别名称'] = df['职位类别名称'].apply(dealStr)
    df['职位要求'] = df['职位要求'].apply(cleanStr)
    df['职位描述'] = df['职位描述'].apply(cleanStr)
    df[['targetjob']] = df.apply( lambda s: pd.Series(target_job(s['职位名称'],s['职位描述'],s['职位要求'],s['发布时间'])), axis = 1)
    df.to_excel('clean.xlsx')
jd_df = get_all_jd('招聘JD')
jd_df.to_excel('all_jd.xlsx')
preprocess(jd_df)

C:\Users\CaiYanWANG\AppData\Local\Temp\ipykernel_18324\3893413127.py:38: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(cell, 'lxml').get_text(separator='')


In [14]:
# 招聘信息汇总
from pathlib import Path
import pandas as pd
import random
from tqdm import tqdm
import datetime

# 暂不需要
def gen_fake(pools: dict, n: int = 300) -> pd.DataFrame:
    rows = []
    for _ in tqdm(range(n), desc="生成假JD"):
        company = random.choice(pools["公司"])
        title   = random.choice(pools["职位名称"])
        city    = random.choice(pools["工作地点"])
        desc = random.choice(pools["职位描述"])
        duty = random.choice(pools["职位要求"])
        job_class = random.choice(pools["职位类别名称"])
        date = random.choice(pools["发布时间"])
        jd_class = random.choice(pools["分类"])

        rows.append({
            "分类":jd_class,
            "职位名称": title,
            "工作地点": city,
            "职位描述": desc,
            "职位要求": duty,
            "发布时间": date,
            "职位类别名称":job_class,
            # 预留标注栏
            "公司": company,
            "label": "",
            "备注": ""
        })
    return pd.DataFrame(rows)


jd_df['发布时间'] = pd.to_datetime(jd_df['发布时间'], errors='coerce')
pool = jd_df[jd_df['发布时间'] >= '2025-06-01'].copy()

# 2. shuffle 后取 300（或全量）
random.seed(42)
sample_300 = pool.sample(n=min(300, len(pool)), random_state=42)

# 3. 加标注栏
sample_300 = sample_300.reset_index(drop=True)
sample_300['label'] = ''        # 0 不考虑 / 1 可面试 / 2 可入职
sample_300['备注']  = ''

# 4. 导出 Excel 直接标
sample_300.to_excel('知识库\JD_val_dataset.xlsx', index=False, engine='openpyxl')
print('已导出 JD_val_dataset.xlsx ，共', len(sample_300), '条，打开就能标 0/1/2')
jd_df.to_excel("知识库\JD_ALL.xlsx", index=False, engine="openpyxl")
print('已导出 jd.xlsx ，共', len(jd_df), '条，打开就能标 0/1/2')


已导出 JD_val_dataset.xlsx ，共 300 条，打开就能标 0/1/2
已导出 jd.xlsx ，共 7412 条，打开就能标 0/1/2


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
招聘知识库质量评估
输入：kb.jsonl 或 kb.csv
输出：quality_report.csv + 7 张 png
"""
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
import warnings, json, os

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ---------- 1. 读库 ----------
def read_kb(path):
    p = Path(path)
    if p.suffix == '.jsonl':
        df = pd.DataFrame(json.loads(l) for l in open(path, encoding='gkb'))
    else:
        df = pd.read_excel(path)
    # 统一小写列名
    df.columns = [c.lower() for c in df.columns]
    return df

# ---------- 2. 基础指标 ----------
def basic_info(df):
    print('===== 基础信息 =====')
    print(f'总岗位数：{len(df)}')
    print(f'字段列表：{list(df.columns)}')
    missing = (df.isnull() | df.astype(str).eq('')).mean() * 100
    print('\n缺失率（%）：')
    for c, v in missing.items():
        print(f'{c:15s} {v:5.1f}')
    return missing

# ---------- 3. 标签分布 & 平衡 ----------
def label_balance(df):
    if 'label' not in df.columns:
        print('\n未发现 label 字段，跳过标签分布')
        return None
    balance = df['label'].value_counts().sort_index()
    print('\n标签分布：')
    print(balance)
    if 0 in balance and 1 in balance:
        print(f'正负比 ≈ {balance[1] / balance[0]:.2f} : 1')
    return balance

# ---------- 4. 技能词频 & 覆盖率 ----------
def skill_analysis(df, topK=30):
    if '技能' not in df.columns:
        print('\n无技能字段，跳过技能分析')
        return
    # 统一 list
    skills = df['skills'].dropna().apply(
        lambda x: x if isinstance(x, list) else str(x).split(','))
    all_words = [w.strip() for sub in skills for w in sub]
    counter = Counter(all_words)
    print(f'\nTop-{topK} 技能词：')
    for w, c in counter.most_common(topK):
        print(f'{w:15s} {c:4d}')
    # 画图
    plt.figure(figsize=(8, 5))

    plt.title(f'Top-{topK} 技能词频')
    plt.savefig('skill_freq.png', dpi=200)
    print('已保存 skill_freq.png')
    return counter

# ---------- 5. 薪资分布 ----------
def salary_dist(df):
    if '薪资下限(k)' not in df.columns or '薪资上限(k)' not in df.columns:
        print('\n无薪资字段，跳过薪资图')
        return
    df['avg_salary'] = (df['薪资下限(k)'] + df['薪资上限(k)']) / 2
    plt.figure(figsize=(6, 4))

    plt.title('平均薪资分布')
    plt.xlabel('k/月')
    plt.savefig('salary_dist.png', dpi=200)
    print('已保存 salary_dist.png')

# ---------- 6. 城市 & 级别 饼图 ----------
def pie_plot(df, col, save_name):
    if col not in df.columns:
        return
    cnt = df[col].value_counts().head(10)
    plt.figure(figsize=(5, 5))
    plt.pie(cnt.values, labels=cnt.index, autopct='%1.1f%%', startangle=90)
    plt.title(f'{col} 分布')
    plt.savefig(save_name, dpi=200)
    print(f'已保存 {save_name}')

# ---------- 7. 描述长度 ----------
def desc_length(df, text_col='描述'):
    if text_col not in df.columns:
        return
    lens = df[text_col].astype(str).str.len()
    print('\n描述长度统计：')
    print(lens.describe())
    plt.figure(figsize=(6, 4))

    plt.title('岗位描述字符数分布')
    plt.savefig('desc_len.png', dpi=200)
    print('已保存 desc_len.png')

# ---------- 8. 标注一致性（可选） ----------
def annotator_consistency(df):
    if '标注人' not in df.columns or df['标注人'].nunique() < 2:
        print('\n不足2名标注人，跳过一致性')
        return
    from sklearn.metrics import cohen_kappa_score
    two = df.groupby('id').head(2)   # 取前两人
    kappa = cohen_kappa_score(two['label'].iloc[::2], two['label'].iloc[1::2])
    print(f'\nCohen Kappa = {kappa:.3f}  （>0.6 可接受，>0.8 优秀）')

# ---------- 9. 主入口 ----------
def main(kb_file='jd.xlsx'):
    df = read_kb(kb_file)
    _ = basic_info(df)
    _ = label_balance(df)
    _ = skill_analysis(df)
    _ = salary_dist(df)
    pie_plot(df, '城市', 'city_pie.png')
    pie_plot(df, '级别', 'level_pie.png')
    _ = desc_length(df)
    annotator_consistency(df)

    # 汇总 CSV
    report = {
    '总岗位': len(df),
    '字段数': len(df.columns),
    '缺失率最大字段': df.isnull().mean().max() * 100,
    '平均薪资': df['avg_salary'].mean() if 'avg_salary' in df else np.nan,
    '中位描述长度': df['描述'].astype(str).str.len().median() if '描述' in df else np.nan,
    '正负比': df['label'].value_counts().sort_index().to_dict() if 'label' in df else np.nan
}
    # 导出
    pd.Series(report, name='value').to_csv('quality_report.csv', header=True)
    print('\n>>> 质量报告已写入 quality_report.csv')
main()

===== 基础信息 =====
总岗位数：6634
字段列表：['分类', '职位名称', '工作地点', '职位描述', '职位要求', '发布时间', '职位类别名称', '公司']

缺失率（%）：
分类                0.0
职位名称              0.0
工作地点              1.0
职位描述              0.0
职位要求              5.3
发布时间             10.2
职位类别名称            4.2
公司                0.0

未发现 label 字段，跳过标签分布

无技能字段，跳过技能分析

无薪资字段，跳过薪资图

不足2名标注人，跳过一致性

>>> 质量报告已写入 quality_report.csv


In [7]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
训练集无 label → 转指令格式（留空）
验证集有 label → 保持原样
"""
import pandas as pd, json, re
from pathlib import Path

def split_skills(req):
    if pd.isna(req):
        return []
    return [w.strip() for w in re.split(r'[,，、；;]', str(req)) if w.strip()]
def build_prompt(row):
    skills = ','.join(row['skills'])
    return (f"职位类别：{row['category']}，"
            f"公司：{row['company']}，"
            f"岗位：{row['job_name']}，"
            f"城市：{row['city']}，"
            f"技能：{skills}，"
            f"描述：{row['desc'][:150]}...")

# ----------- 主入口 -----------
def convert(in_file, out_file, has_label=False):
    df = pd.read_excel(in_file) if str(in_file).endswith('.xlsx') else pd.read_csv(in_file)

    # ① 按“你真实列”→ 标准名 映射，不强制 9 列
    col_map = {
        '分类': 'category',
        '职位名称': 'job_name',
        '工作地点': 'city',
        '职位描述': 'desc',
        '职位要求': 'require',
        '发布时间': 'publish_time',
        '职位类别名称': 'pos_category',
        '公司': 'company'
    }
    df = df.rename(columns=col_map)

    # ② 若无 skills，现拆
    df['skills'] = df['require'].apply(split_skills)

    # ③ 写文件
    with open(out_file, 'w', encoding='utf-8') as f:
        for _, r in df.iterrows():
            output = (f"{int(r['label'])}分，"
                      f"{'可入职' if r['label']==2 else '可面试' if r['label']==1 else '不考虑'}"
                     ) if has_label else ""
            rec = {
                "instruction": "你是一个智能招聘助手，请根据用户偏好判断TA是否会喜欢该岗位。",
                "input": build_prompt(r),
                "output": output
            }
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')
    print(f'>>> 已生成 {out_file}  共 {len(df)} 条')
# ---------------- 调用 ----------------
# 1. 训练集无 label
convert('jd.xlsx', 'train.jsonl', has_label=False)

# 2. 验证集有 label
convert('to_label_300.xlsx',   'val.jsonl',   has_label=True)

>>> 已生成 train.jsonl  共 6634 条
>>> 已生成 val.jsonl  共 301 条


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
7-Dimension Data Quality Score
usage: python dq_score.py  # 默认读 raw_corpus.csv，输出 clean_corpus.csv
"""

import re, hashlib, math, json, tqdm, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# ================== 1. 参数 ==================
MAX_LEN      = 2048
MIN_LEN      = 10
NON_ZH_RATIO = 0.25          # 允许最大非中文字符比
AD_KEYWORDS  = {'加微信', '红包', '内推', '扫码', '优惠券', '加我', 'q群'}
KEYWORDS = {
    # ----- 大模型 LLM -----
    'llm', 'gpt', 'bert', 'transformer', 'attention', 'self_attention',
    'lora', 'qlora', 'peft', 'fine_tune', 'sft', 'rlhf', 'instruct',
    'token', 'embedding', 'positional_encoding', 'ffn', 'layer_norm',
    'dropout', 'softmax', 'gelu', 'relu', 'sigmoid', 'adamw', 'adam',
    'scheduler', 'warmup', 'batch_size', 'gradient_accumulation',
    'mixed_precision', 'fp16', 'bf16', 'tensor_parallel', 'pipeline_parallel',
    'data_parallel', 'deepspeed', 'zero', 'offload', 'checkpoint',
    'kv_cache', 'flash_attention', 'rope', 'alibi', 'gpt3', 'gpt4',
    'chatglm', 'baichuan', 'llama', 'alpaca', 'vicuna', 'falcon',
    'prompt', 'prompt_template', 'few_shot', 'zero_shot', 'chain_of_thought',
    'rag', 'retriever', 'index', 'faiss', 'milvus', 'embedding_model',
    'cosine_similarity', 'rerank', 'cross_encoder', 'generation',
    'temperature', 'top_p', 'top_k', 'beam_search', 'do_sample',
    'hallucination', 'perplexity', 'bleu', 'rouge', 'distinct',
    'reward_model', 'critic', 'actor', 'ppo', 'dpo', 'kto',

    # ----- 数据仓库 & 大数据 -----
    'data_warehouse', 'dwd', 'dws', 'ads', 'dim', 'ods', 'etl', 'elt',
    'sql', 'hive', 'spark', 'spark_sql', 'pyspark', 'flink', 'kafka',
    'hadoop', 'hdfs', 'yarn', 'mapreduce', 'tez', 'impala', 'presto',
    'clickhouse', 'doris', 'starrocks', 'hbase', 'redis', 'mongodb',
    'mysql', 'postgresql', 'oracle', 'sqlserver', 'tidb', 'oceanbase',
    'data_lake', 'lakehouse', 'iceberg', 'hudi', 'delta_lake', 'paimon',
    'binlog', 'cdc', 'canal', 'debezium', 'airflow', 'dolphinscheduler',
    'azkaban', 'oozie', 'crontab', 'shell', 'git', 'gitlab', 'github',
    'parquet', 'orc', 'avro', 'json', 'csv', 'tsv', 'xml', 'protobuf',
    'partition', 'bucket', 'shuffle', 'broadcast', 'sort_merge',
    'cube', 'rollup', 'grouping_sets', 'window_function', 'rank', 'row_number',
    'lag', 'lead', 'udf', 'udaf', 'udtf', 'hive_function', 'spark_function',
    'slow_query', 'explain', 'index', 'btree', 'hash_index', 'columnar',
    'star_schema', 'snowflake_schema', 'fact_table', 'dimension_table',
    'surrogate_key', 'natural_key', 'scd', 'scd_type2', 'cdc_merge',
    'data_quality', 'data_profiling', 'anomaly_detection', 'completeness',
    'consistency', 'timeliness', 'uniqueness', 'validity', 'accuracy',
    'lineage', 'impact_analysis', 'data_governance', 'metadata',
    'atlas', 'datahub', 'metacat', 'griffin', 'great_expectations',
    'dbt', 'sqlmesh', 'looker', 'tableau', 'superset', 'quick_bi',
    'aliyun_maxcompute', 'aws_redshift', 'gcp_bigquery', 'azure_synapse',
    'starrocks_connector', 'flink_connector', 'kafka_connector',
    '大模型', '大语言模型', '预训练', '微调', '指令微调', '人类反馈', '强化学习', 'LoRA', 'QLoRA',
    '提示词', '提示模板', '少样本', '零样本', '思维链', '上下文', 'token', '嵌入', '词向量',
    '位置编码', '注意力', '自注意力', '多头注意力', '前馈网络', '层归一化', ' dropout',
    'softmax', '激活函数', 'AdamW', '学习率', '预热', '梯度累积', '混合精度', 'FP16', 'BF16',
    '张量并行', '流水线并行', '数据并行', 'DeepSpeed', 'ZeRO', '检查点', 'KV缓存', 'FlashAttention',
    'RoPE', 'ALiBi', 'GPT', 'ChatGLM', 'Baichuan', 'LLaMA', 'Alpaca', 'Vicuna',
    '温度采样', 'Top_P', 'Top_K', '束搜索', '幻觉', '困惑度', 'BLEU', 'ROUGE',
    '奖励模型', 'PPO', 'DPO', 'RAG', '检索器', '向量库', 'Faiss', 'Milvus', '重排',
    '数据仓库', '数仓', 'ODS', 'DWD', 'DWS', 'ADS', '维度表', '事实表', '缓慢变化维',
    'ETL', 'ELT', '数据集成', '数据清洗', '数据建模', '星型模型', '雪花模型',
    'Hive', 'Spark', 'Flink', 'Kafka', 'Hadoop', 'HDFS', 'YARN', 'MapReduce',
    'ClickHouse', 'Doris', 'StarRocks', 'HBase', 'Redis', 'MySQL', 'PostgreSQL',
    '数据湖', '湖仓一体', 'Iceberg', 'Hudi', 'DeltaLake', 'Paimon', 'Binlog', 'CDC',
    'Airflow', '海豚调度', '数据质量', '数据治理', '元数据', '数据血缘', '主键',
    '分区', '分桶', '索引', '布隆过滤器', '列式存储', 'Parquet', 'ORC', 'Avro',
    '窗口函数', '排名', '累计和', 'UDF', 'UDAF', 'UDTF', 'Explain', '执行计划',
    '慢查询', '数据倾斜', 'Shuffle', '广播', '排序合并', 'Cube', 'Rollup',
    '一致性', '完整性', '唯一性', '及时性', '有效性', '异常检测', '数据探查',
    'DBT', 'SQLMesh', 'Tableau', 'QuickBI', 'MaxCompute', 'BigQuery', 'Redshift'
}   # 可自行扩充
WEIGHT = {'len':0.10, 'encode':0.10, 'ad':0.10, 'dup':0.30,
          'sem':0.15, 'kw':0.15, 'fmt':0.10}  # 权重和=1
THRESHOLD = 0.75            # 高质量线
MODEL = SentenceTransformer("ernie-3.0-base-zh")  # 75 MB，CPU 可跑

# ================== 2. 工具函数 ==================
def simhash_hash(text):
    v = [0]*64
    for tok in text.split():
        h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
        for i in range(64):
            v[i] += 1 if (h >> i) & 1 else -1
    return sum(1 << i for i in range(64) if v[i] > 0)

def hamming_dist(h1, h2):
    return bin(h1 ^ h2).count('1')

# ------------------ 各维度打分 ------------------
def len_score(texts):
    return texts.apply(lambda x: 1 if MIN_LEN <= len(x) <= MAX_LEN else 0)

def encode_score(texts):
    def _enc(x):
        zh = len(re.findall(r'[\u4e00-\u9fff]', x))
        ratio = (len(x) - zh) / len(x) if len(x) else 1
        return max(0, 1 - ratio / NON_ZH_RATIO) if ratio < NON_ZH_RATIO else 0
    return texts.apply(_enc)

def ad_score(texts):
    return texts.apply(lambda x: 0 if any(k in x for k in AD_KEYWORDS) else 1)

def dup_score(texts, win=100, ham=3):
    hashes = texts.apply(simhash_hash)
    scores = []
    for i, h in enumerate(hashes):
        dup = any(hamming_dist(h, hashes[j]) < ham for j in range(max(0, i-win), i))
        scores.append(0 if dup else 1)
    return pd.Series(scores, index=texts.index)

def sem_score(texts, batch=256):
    embs = MODEL.encode(texts.tolist(), batch_size=batch, normalize_embeddings=True, show_progress_bar=False)
    center = embs.mean(axis=0, keepdims=True)
    cos = cosine_similarity(embs, center).squeeze()
    return (cos - cos.min()) / (cos.max() - cos.min() + 1e-8)

def kw_score(texts):
    vectorizer = TfidfVectorizer(vocabulary=KEYWORDS, token_pattern=r'(?u)\b\w+\b')
    tfidf = vectorizer.fit_transform(texts)
    scores = [np.sort(row.data)[-10:].sum() for row in tfidf]
    scores = np.array(scores)
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

def fmt_score(texts):
    # 简易格式：必须含问号且 15<=len<=200
    return texts.apply(lambda x: 1 if ('?' in x or '？' in x) and 15<=len(x)<=200 else 0)

# ================== 3. 主函数 ==================
def dq_score(df, text_col='text'):
    df = df.copy()
    texts = df[text_col].astype(str)

    ls = len_score(texts)
    es = encode_score(texts)
    ad = ad_score(texts)
    dp = dup_score(texts)
    sm = sem_score(texts)
    kw = kw_score(texts)
    ft = fmt_score(texts)

    df['dq_score'] = (WEIGHT['len']*ls + WEIGHT['encode']*es +
                      WEIGHT['ad']*ad + WEIGHT['dup']*dp +
                      WEIGHT['sem']*sm + WEIGHT['kw']*kw + WEIGHT['fmt']*ft)
    return df

# ================== 4. IO & 可视化 ==================
def main():
    in_file  = 'raw_corpus.csv'   # 需含列 text
    out_file = 'clean_corpus.csv'

    df = pd.read_csv(in_file)
    print('>>> 原始数据:', len(df))
    df = dq_score(df)
    good = df[df.dq_score >= THRESHOLD].copy()
    print('>>> 高质量数据:', len(good), f'保留率 {len(good)/len(df):.1%}')
    good.to_csv(out_file, index=False)

    # 画分布
    plt.hist(df['dq_score'], bins=50, alpha=0.7, label='all')
    plt.hist(good['dq_score'], bins=50, alpha=0.7, label='clean')
    plt.axvline(THRESHOLD, color='red', linestyle='--')
    plt.legend(); plt.xlabel('DQ Score'); plt.ylabel('Count')
    plt.title('Data Quality Distribution')
    plt.savefig('dq_dist.png', dpi=200); plt.close()
    print('>>> 分布图已保存：dq_dist.png')

if __name__ == '__main__':
    main()